In [0]:
hchb_system_settings = dbutils.widgets.get("hchb_system_settings")
hchb_hard_close_periods = dbutils.widgets.get("hchb_hard_close_periods")
hchb_branches = dbutils.widgets.get("hchb_branches")
hchb_agencies = dbutils.widgets.get("hchb_agencies")
hchb_service_lines = dbutils.widgets.get("hchb_service_lines")
hchb_payor_types = dbutils.widgets.get("hchb_payor_types")
hchb_payor_sources = dbutils.widgets.get("hchb_payor_sources")
hchb_financial_class = dbutils.widgets.get("hchb_financial_class")
hchb_v_hard_close_header_info = dbutils.widgets.get("hchb_v_hard_close_header_info")
hchb_hard_close_headers = dbutils.widgets.get("hchb_hard_close_headers")
hchb_hard_close_revenue = dbutils.widgets.get("hchb_hard_close_revenue")
hchb_vi_hard_close_manual_adjustments = dbutils.widgets.get("hchb_vi_hard_close_manual_adjustments")
hchb_hard_close_cash = dbutils.widgets.get("hchb_hard_close_cash")
hchb_hard_close_credits = dbutils.widgets.get("hchb_hard_close_credits")
hchb_clients_all = dbutils.widgets.get("hchb_clients_all")
hchb_hard_close_pdgm_headers = dbutils.widgets.get("hchb_hard_close_pdgm_headers")
hchb_pdgm_period = dbutils.widgets.get("hchb_pdgm_period")
hchb_client_episode_fs = dbutils.widgets.get("hchb_client_episode_fs")
hchb_hard_close_pdgm_revenue = dbutils.widgets.get("hchb_hard_close_pdgm_revenue")
hchb_hard_close_pdgm_cash = dbutils.widgets.get("hchb_hard_close_pdgm_cash")
hchb_hard_close_pdgm_credits = dbutils.widgets.get("hchb_hard_close_pdgm_credits")
hchb_hard_close_nonpps_balance_snapshot = dbutils.widgets.get("hchb_hard_close_nonpps_balance_snapshot")
hchb_hard_close_nonpps_balance_snapshot_days = dbutils.widgets.get("hchb_hard_close_nonpps_balance_snapshot_days")
hchb_hard_close_nonpps_headers = dbutils.widgets.get("hchb_hard_close_nonpps_headers")
tempdb_dbo_hchbar = dbutils.widgets.get("tempdb_dbo_hchbar")

In [0]:
# Truncate the temp table
spark.sql(f"TRUNCATE TABLE {tempdb_dbo_hchbar}")

print(f"✓ Table {tempdb_dbo_hchbar} truncated successfully")

In [0]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# Set Spark SQL configurations
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

print("=" * 60)
print("AR REPORT PARAMETER INITIALIZATION")
print("=" * 60)

# =============================================
# SECTION 1: Parameter Initialization
# =============================================

print("\nSection 1: Calculating reporting dates...")

# Calculate reporting dates in Python
today = datetime.now().date()

# Calculate week ending date (Sunday)
# Python weekday(): Monday=0, Sunday=6
# Spark DAYOFWEEK: Sunday=1, Monday=2, ..., Saturday=7
day_of_week = today.weekday()  # 0=Monday, 6=Sunday
if day_of_week == 6:  # Sunday
    reporting_week_ending_date = today
else:
    # Go back to previous Sunday
    days_back = (day_of_week + 1) % 7
    reporting_week_ending_date = today - timedelta(days=days_back)

print(f"Today: {today}")
print(f"Reporting Week Ending Date: {reporting_week_ending_date}")

# Calculate reporting month and year
if reporting_week_ending_date.day <= 3:
    # Use prior month
    temp_date = reporting_week_ending_date.replace(day=1) - timedelta(days=1)
    reporting_week_ending_month_number = temp_date.month
    reporting_week_ending_year_number = temp_date.year
    # Get last day of prior month
    reporting_month_end = temp_date
else:
    # Use current month
    reporting_week_ending_month_number = reporting_week_ending_date.month
    reporting_week_ending_year_number = reporting_week_ending_date.year
    # Get last day of current month
    next_month = reporting_week_ending_date.replace(day=28) + timedelta(days=4)
    reporting_month_end = next_month - timedelta(days=next_month.day)

print(f"Reporting Month End: {reporting_month_end}")
print(f"Reporting Period: {reporting_week_ending_year_number}{reporting_week_ending_month_number:02d}")

# =============================================
# SECTION 2: Define Report Parameters
# =============================================

print("\nSection 2: Setting report parameters...")

# Calculate closing period
closing_period = (reporting_week_ending_year_number * 100) + reporting_week_ending_month_number
cash_received_through = closing_period

# Define all report parameters as Python variables
aging_date = datetime.strptime('2025-06-29', '%Y-%m-%d').date()
ptid = None
aging_filter = None
snapshot = 0  # Deprecated parameter
age_from_soe = 0  # 0 = END of episode, 1 = Start of episode
branch_list = None
service_line_list = None
agency_list = None
exclude_lower = '-0.3'
exclude_upper = '0.3'
rptreqrpt_id = 0  # Report ID
psid = None  # Payor Source ID
balance_filter = 0  # 0 = none, 1 = Credits/overpayments, 2 = debit balances, 3 = use exclude number range
group_by = 0  # No longer used
show_results = 'N'
balance_determined_at = 3
period_end_date = reporting_month_end

print(f"Aging Date: {aging_date}")
print(f"Closing Period: {closing_period}")
print(f"Cash Received Through: {cash_received_through}")
print(f"Period End Date: {period_end_date}")

# =============================================
# SECTION 3: System Settings
# =============================================

print("\nSection 3: Getting system settings...")

# Get PDGM Future Period Unearned setting
pdgm_result = spark.sql(f"""
SELECT COALESCE(
    (SELECT ss_Value 
     FROM {hchb_system_settings}
     WHERE ss_Setting = 'PdgmShowFuturePeriodUnearned' 
     LIMIT 1),
    'N'
) AS PdgmShowFuturePeriodUnearned
""")

pdgm_show_future_period_unearned = pdgm_result.collect()[0]['PdgmShowFuturePeriodUnearned']
print(f"PDGM Show Future Period Unearned: {pdgm_show_future_period_unearned}")

# =============================================
# SECTION 4: Aging Bucket Configuration
# =============================================

print("\nSection 4: Configuring aging buckets...")

export_option = 1  # 0 = standard; 1 = One Year; 2 = Two Year

# Calculate aging buckets
a0 = aging_filter if aging_filter is not None else 0
a1 = a0 + 30 if export_option == 0 else 30
a2 = a0 + 60 if export_option == 0 else 60
a3 = a0 + 90 if export_option == 0 else 90
a4 = a0 + 120 if export_option == 0 else 120

if export_option == 0:
    a5 = a0 + 150
elif export_option == 1:
    a5 = 180
else:
    a5 = 150

if export_option == 0:
    a6 = 9999
elif export_option == 1:
    a6 = 270
else:
    a6 = 180

if export_option == 0:
    a7 = 9999
elif export_option == 1:
    a7 = 365
else:
    a7 = 240

a8 = 9999 if export_option == 0 else 365
a9 = 9999 if export_option == 0 else 531
a10 = 730

print(f"Aging buckets: a0={a0}, a1={a1}, a2={a2}, a3={a3}, a4={a4}, a5={a5}")
print(f"              a6={a6}, a7={a7}, a8={a8}, a9={a9}, a10={a10}")

# Create Aging Labels
aging_label1 = f"{a0} - {a1}"
aging_label2 = f"{a1 + 1} - {a2}"
aging_label3 = f"{a2 + 1} - {a3}"
aging_label4 = f"{a3 + 1} - {a4}"
aging_label5 = f"{a4 + 1}+" if export_option == 0 else f"{a4 + 1} - {a5}"
aging_label6 = f"{a5 + 1} - {a6}"
aging_label7 = f"{a6 + 1} - {a7}"
aging_label8 = f"{a7 + 1}+" if export_option == 1 else f"{a7 + 1} - {a8}"
aging_label9 = f"{a8 + 1} - {a9}"
aging_label10 = f"{a9 + 1} - {a10}"
aging_label11 = f"{a10 + 1}+"

print(f"\nAging Labels:")
print(f"  {aging_label1}, {aging_label2}, {aging_label3}, {aging_label4}, {aging_label5}")

# =============================================
# SECTION 5: Get Max Period Date
# =============================================

print("\nSection 5: Getting max period date...")

max_period_result = spark.sql(f"""
SELECT hcp.hcp_EndDate AS MaxPeriodDate
FROM {hchb_hard_close_periods} hcp
WHERE hcp.hcp_Period = {closing_period}
""")

if max_period_result.count() > 0:
    max_period_date = max_period_result.collect()[0]['MaxPeriodDate']
    print(f"Max Period Date: {max_period_date}")
else:
    max_period_date = None
    print("Max Period Date: Not found")

# =============================================
# SECTION 6: Parameter Filter Tables
# =============================================

print("\nSection 6: Creating parameter filter tables...")

# Branches Filter
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Branches_Filter AS
SELECT Branch_Code AS BranchCode
FROM {hchb_branches}
""")

# Agencies Filter
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Agencies_Filter AS
SELECT 
    a.Agency_id AS agid,
    CONCAT(a.Agency_name, ':', a.Agency_ProviderNumber) AS Agency
FROM {hchb_agencies} a
""")

# Service Lines Filter
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ServiceLines_Filter AS
SELECT sl_id AS ID
FROM {hchb_service_lines}
""")

# Payor Types Filter
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW PayorTypes_Filter AS
SELECT pt_id AS ID
FROM {hchb_payor_types}
""")

# Payor Sources Filter
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW PayorSources_Filter AS
SELECT ps_id AS ID
FROM {hchb_payor_sources}
""")

# Financial Classes Filter
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW FinancialClasses_Filter AS
SELECT fc.fc_id AS ID
FROM {hchb_financial_class} fc
WHERE EXISTS (
    SELECT 1 
    FROM {hchb_system_settings}
    WHERE ss_setting = 'EnableFinancialClass' 
    AND ss_value = 'N'
)
""")

print("✓ Filter tables created")

# =============================================
# SECTION 7: Exclude Range Parameters
# =============================================

print("\nSection 7: Processing exclude range parameters...")

# Process exclude parameters
try:
    p_exclude_lower = float(exclude_lower)
except:
    p_exclude_lower = 0.0

try:
    p_exclude_upper = float(exclude_upper)
except:
    p_exclude_upper = 0.0

print(f"Exclude Lower: {p_exclude_lower}")
print(f"Exclude Upper: {p_exclude_upper}")

# =============================================
# SUMMARY
# =============================================

print("\n" + "=" * 60)
print("PARAMETER INITIALIZATION COMPLETED")
print("=" * 60)
print(f"\nKey Parameters:")
print(f"  Reporting Week Ending: {reporting_week_ending_date}")
print(f"  Closing Period: {closing_period}")
print(f"  Aging Date: {aging_date}")
print(f"  Period End Date: {period_end_date}")
print(f"  Age From SOE: {'Start' if age_from_soe else 'End'}")
print(f"  Export Option: {export_option}")
print(f"\nFilter Counts:")

# Get counts of filter records
for filter_name in ['Branches_Filter', 'Agencies_Filter', 'ServiceLines_Filter', 
                     'PayorTypes_Filter', 'PayorSources_Filter', 'FinancialClasses_Filter']:
    count = spark.sql(f"SELECT COUNT(*) as count FROM {filter_name}").collect()[0]['count']
    print(f"  {filter_name}: {count} records")

print("\n✓ Ready for AR report generation")

In [0]:
# =============================================
# SECTION 1: Get Max Hard Close Headers
# =============================================

print("\nSection 1: Getting max hard close headers...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW MaxHCH AS
SELECT
    hch_AuthID AS AuthID,
    hch_msp AS MSP,
    hch_fcid AS FCID,
    MAX(hch_id) AS HCHID
FROM {hchb_hard_close_headers}
WHERE hch_Closing_Period BETWEEN 0 AND {closing_period}
GROUP BY hch_AuthID, hch_msp, hch_fcid
""")

max_hch_count = spark.sql("SELECT COUNT(*) as count FROM MaxHCH").collect()[0]['count']
print(f"✓ Max HCH records: {max_hch_count}")

# =============================================
# SECTION 2: Get Max Hard Close Header Details
# =============================================

print("\nSection 2: Getting max hard close header details...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW MaxHCHRow AS
SELECT
    v.hch_id,
    v.hch_AuthID,
    v.hch_AuthStartDate,
    v.hch_AuthEndDate,
    v.hch_Closing_Period,
    v.hch_Closing_BranchCode,
    v.hch_InsuredId,
    v.hch_paid,
    v.hch_LastName,
    v.hch_FirstName,
    v.hch_mi,
    v.hch_ptid,
    CONCAT(v.hch_ptDesc, CASE WHEN v.hch_msp = TRUE THEN '(MSP)' ELSE '' END) AS hch_ptDesc,
    v.hch_msp,
    v.hch_slid,
    v.hch_slDesc,
    v.hch_fcid,
    v.hch_fcDesc,
    v.hch_psid,
    v.hch_psDesc,
    v.hch_AgencyId,
    v.hch_AgencyName,
    v.hch_ProviderNumber,
    v.hch_EOEType,
    v.hch_BillDate,
    v.hch_epiid,
    v.hch_AdmitDate,
    v.hch_DischargeDate
FROM MaxHCH mh
JOIN {hchb_v_hard_close_header_info} v 
    ON v.hch_id = mh.HCHID 
    AND v.hch_msp = mh.MSP
""")

max_hch_row_count = spark.sql("SELECT COUNT(*) as count FROM MaxHCHRow").collect()[0]['count']
print(f"✓ Max HCH Row records: {max_hch_row_count}")

# =============================================
# SECTION 3: PPS Earned Revenue, Adjustments, and Unearned Revenue
# =============================================

print("\nSection 3: Calculating PPS revenue, adjustments, and unearned revenue...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ReportDetailsPPS_Revenue AS
SELECT
    1 AS PPS,
    h.hch_AuthID AS KeyID,
    h.hch_fcid AS FCID,
    h.hch_ptid AS PTID,
    h.hch_psid AS PSID,
    h.hch_msp AS MSP,
    h.hch_slid AS SLID,
    h.hch_AgencyId AS AgencyId,
    hcr.hcr_Reporting_BranchCode AS Reporting_BranchCode,
    SUM(CASE WHEN hcr.hcr_type = 'E' THEN COALESCE(hcr.hcr_amount, 0) ELSE 0 END) AS EarnedRev,
    SUM(CASE WHEN hcr.hcr_type = 'A' THEN COALESCE(hcr.hcr_amount, 0) ELSE 0 END) AS Adjustments,
    SUM(CASE WHEN hcr.hcr_type = 'U' AND mh.AuthID IS NOT NULL THEN COALESCE(hcr.hcr_amount, 0) ELSE 0 END) AS UnEarnedRev,
    0 AS Cash,
    0 AS Credits,
    0 AS Refunds
FROM {hchb_hard_close_revenue} hcr
JOIN {hchb_hard_close_headers} h1 ON h1.hch_id = hcr.hcr_hchid
JOIN MaxHCHRow h ON h.hch_AuthID = h1.hch_AuthID AND h.hch_msp = h1.hch_msp AND h.hch_fcid = h1.hch_fcid
JOIN Branches_Filter b ON b.BranchCode = hcr.hcr_Reporting_BranchCode
JOIN ServiceLines_Filter sl ON sl.ID = h.hch_slid
JOIN Agencies_Filter ag ON ag.agid = h.hch_AgencyId
JOIN PayorTypes_Filter pt ON pt.ID = h.hch_ptid
JOIN PayorSources_Filter ps ON ps.ID = h.hch_psid
JOIN FinancialClasses_Filter fc ON fc.ID = h.hch_fcid
LEFT JOIN MaxHCH mh ON mh.HCHID = h1.hch_id
WHERE h1.hch_Closing_Period <= {closing_period}
GROUP BY 
    h.hch_AuthID, h.hch_msp, h.hch_ptid, h.hch_psid, h.hch_slid, 
    h.hch_AgencyId, hcr.hcr_Reporting_BranchCode, h.hch_fcid
""")

revenue_count = spark.sql("SELECT COUNT(*) as count FROM ReportDetailsPPS_Revenue").collect()[0]['count']
print(f"✓ PPS Revenue records: {revenue_count}")

# =============================================
# SECTION 4: PPS Manual Adjustments
# =============================================

print("\nSection 4: Processing PPS manual adjustments...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ReportDetailsPPS_ManualAdj AS
SELECT
    1 AS PPS,
    h.hch_AuthID AS KeyID,
    h.hch_fcid AS FCID,
    h.hch_ptid AS PTID,
    h.hch_psid AS PSID,
    h.hch_msp AS MSP,
    h.hch_slid AS SLID,
    h.hch_AgencyId AS AgencyId,
    m.Reporting_BranchCode AS Reporting_BranchCode,
    0 AS EarnedRev,
    SUM(CASE WHEN m.reporting_period <= {closing_period} THEN COALESCE(m.hcma_amount, 0) ELSE 0 END) AS Adjustments,
    SUM(CASE WHEN m.reporting_period > {closing_period} THEN COALESCE(m.hcma_amount, 0) ELSE 0 END) AS UnEarnedRev,
    0 AS Cash,
    0 AS Credits,
    0 AS Refunds
FROM {hchb_vi_hard_close_manual_adjustments} m
JOIN {hchb_hard_close_headers} h1 ON h1.hch_id = m.hch_id
JOIN MaxHCHRow h ON h.hch_AuthID = h1.hch_AuthID AND h.hch_msp = h1.hch_msp AND h.hch_fcid = h1.hch_fcid
JOIN Branches_Filter b ON b.BranchCode = m.Reporting_BranchCode
JOIN ServiceLines_Filter sl ON sl.ID = h.hch_slid
JOIN Agencies_Filter ag ON ag.agid = h.hch_AgencyId
JOIN PayorTypes_Filter pt ON pt.ID = h.hch_ptid
JOIN PayorSources_Filter ps ON ps.ID = h.hch_psid
JOIN FinancialClasses_Filter fc ON fc.ID = h.hch_fcid
WHERE m.hch_Closing_Period <= {closing_period}
GROUP BY 
    h.hch_AuthID, h.hch_msp, h.hch_ptid, h.hch_psid, h.hch_slid, 
    h.hch_AgencyId, m.Reporting_BranchCode, h.hch_fcid
""")

manual_adj_count = spark.sql("SELECT COUNT(*) as count FROM ReportDetailsPPS_ManualAdj").collect()[0]['count']
print(f"✓ PPS Manual Adjustment records: {manual_adj_count}")

# =============================================
# SECTION 5: PPS Cash
# =============================================

print("\nSection 5: Processing PPS cash receipts...")

# Handle NULL cash_received_through
cash_received_through_condition = f"""
(
    (h1.hch_Closing_Period <= {closing_period} AND {cash_received_through} IS NULL)
    OR ({cash_received_through} IS NOT NULL AND h1.hch_Closing_Period <= {cash_received_through})
)
""" if cash_received_through is not None else f"h1.hch_Closing_Period <= {closing_period}"

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ReportDetailsPPS_Cash AS
SELECT
    1 AS PPS,
    h.hch_AuthID AS KeyID,
    h.hch_fcid AS FCID,
    h.hch_ptid AS PTID,
    h.hch_psid AS PSID,
    h.hch_msp AS MSP,
    h.hch_slid AS SLID,
    h.hch_AgencyId AS AgencyId,
    hcc.hcc_Reporting_BranchCode AS Reporting_BranchCode,
    0 AS EarnedRev,
    0 AS Adjustments,
    0 AS UnEarnedRev,
    SUM(COALESCE(hcc.hcc_amount, 0)) AS Cash,
    0 AS Credits,
    0 AS Refunds
FROM {hchb_hard_close_cash} hcc
JOIN {hchb_hard_close_headers} h1 ON h1.hch_id = hcc.hcc_hchid
JOIN MaxHCHRow h ON h.hch_AuthID = h1.hch_AuthID AND h.hch_msp = h1.hch_msp AND h.hch_fcid = h1.hch_fcid
JOIN Branches_Filter b ON b.BranchCode = hcc.hcc_Reporting_BranchCode
JOIN ServiceLines_Filter sl ON sl.ID = h.hch_slid
JOIN Agencies_Filter ag ON ag.agid = h.hch_AgencyId
JOIN PayorTypes_Filter pt ON pt.ID = h.hch_ptid
JOIN PayorSources_Filter ps ON ps.ID = h.hch_psid
JOIN FinancialClasses_Filter fc ON fc.ID = h.hch_fcid
WHERE {cash_received_through_condition}
GROUP BY 
    h.hch_AuthID, h.hch_msp, h.hch_ptid, h.hch_psid, h.hch_slid, 
    h.hch_AgencyId, hcc.hcc_Reporting_BranchCode, h.hch_fcid
""")

cash_count = spark.sql("SELECT COUNT(*) as count FROM ReportDetailsPPS_Cash").collect()[0]['count']
print(f"✓ PPS Cash records: {cash_count}")

# =============================================
# SECTION 6: Combine PPS Details (Excluding Credits/Refunds)
# =============================================

print("\nSection 6: Combining PPS details...")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW ReportDetailsPPS_Combined AS
SELECT
    PPS, KeyID, FCID, PTID, PSID, MSP, SLID, AgencyId, Reporting_BranchCode,
    SUM(EarnedRev) AS EarnedRev,
    SUM(Adjustments) AS Adjustments,
    SUM(UnEarnedRev) AS UnEarnedRev,
    SUM(Cash) AS Cash,
    SUM(Credits) AS Credits,
    SUM(Refunds) AS Refunds
FROM (
    SELECT * FROM ReportDetailsPPS_Revenue
    UNION ALL
    SELECT * FROM ReportDetailsPPS_ManualAdj
    UNION ALL
    SELECT * FROM ReportDetailsPPS_Cash
)
GROUP BY PPS, KeyID, FCID, PTID, PSID, MSP, SLID, AgencyId, Reporting_BranchCode
""")

combined_count = spark.sql("SELECT COUNT(*) as count FROM ReportDetailsPPS_Combined").collect()[0]['count']
print(f"✓ PPS Combined records: {combined_count}")

# =============================================
# SECTION 7: PPS Report Data (Main Report)
# =============================================

print("\nSection 7: Creating PPS main report...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_MonthEndClose_PPS AS
SELECT
    r.PPS,
    r.KeyID,
    m.hch_AuthStartDate AS KeyDate,
    m.hch_AuthEndDate AS EndDate,
    m.hch_paid AS PAID,
    CONCAT(COALESCE(m.hch_LastName, ''), CASE WHEN m.hch_LastName IS NOT NULL THEN ', ' ELSE '' END, COALESCE(m.hch_FirstName, '')) AS ClientName,
    r.FCID,
    m.hch_fcDesc AS Financial_Class,
    r.PTID,
    m.hch_ptDesc AS Payor_Type,
    r.PSID,
    m.hch_psDesc AS Payor_Source,
    r.MSP,
    r.SLID,
    m.hch_slDesc AS slDesc,
    r.AgencyId,
    CONCAT(m.hch_AgencyName, ':', m.hch_ProviderNumber) AS Agency,
    m.hch_InsuredId AS InsuredId,
    m.hch_EOEType AS EOEType,
    m.hch_BillDate AS BillDate,
    r.Reporting_BranchCode,
    SUM(COALESCE(r.EarnedRev, 0)) AS EarnedRev,
    SUM(COALESCE(r.Adjustments, 0)) AS Adjustments,
    SUM(COALESCE(r.UnEarnedRev, 0)) AS UnEarnedRev,
    SUM(COALESCE(r.Cash, 0)) AS Cash,
    SUM(COALESCE(r.Credits, 0)) AS Credits,
    SUM(COALESCE(r.Refunds, 0)) AS Refunds,
    MAX(CASE 
        WHEN DATEDIFF(m.hch_BillDate, m.hch_AuthEndDate) < 0 THEN 0 
        ELSE COALESCE(DATEDIFF(m.hch_BillDate, m.hch_AuthEndDate), 0) 
    END) AS DaysDelayed,
    '{period_end_date}' AS LastDayClosingPeriod,
    MAX(COALESCE(
        CASE 
            WHEN {age_from_soe} = 1 THEN DATEDIFF('{period_end_date}', m.hch_AuthStartDate)
            ELSE DATEDIFF('{period_end_date}', m.hch_AuthEndDate)
        END, -1
    )) AS AgingDays,
    CAST(NULL AS BIGINT) AS RevenueDetailID,
    0 AS TCID
FROM ReportDetailsPPS_Combined r
JOIN MaxHCHRow m ON m.hch_AuthID = r.KeyID AND m.hch_msp = r.MSP AND m.hch_fcid = r.FCID
GROUP BY 
    r.PPS, r.KeyID, m.hch_AuthStartDate, m.hch_AuthEndDate, m.hch_paid,
    r.FCID, m.hch_fcDesc, r.PTID, m.hch_ptDesc, r.PSID, m.hch_psDesc, r.MSP,
    r.SLID, m.hch_slDesc, r.AgencyId, m.hch_AgencyName, m.hch_ProviderNumber, m.hch_FirstName, m.hch_LastName,
    m.hch_InsuredId, m.hch_EOEType, m.hch_BillDate, r.Reporting_BranchCode
""")

report_count = spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PPS").collect()[0]['count']
print(f"✓ PPS Report records: {report_count}")

# =============================================
# SUMMARY
# =============================================

print("\n" + "=" * 60)
print("PPS DATA RETRIEVAL COMPLETED")
print("=" * 60)

summary = spark.sql("""
SELECT 
    COUNT(*) as TotalRecords,
    SUM(EarnedRev) as TotalEarnedRev,
    SUM(Adjustments) as TotalAdjustments,
    SUM(UnEarnedRev) as TotalUnEarnedRev,
    SUM(Cash) as TotalCash,
    COUNT(DISTINCT PAID) as UniqueClients,
    COUNT(DISTINCT Reporting_BranchCode) as UniqueBranches
FROM Report_MonthEndClose_PPS
""")

summary.show(truncate=False)

# Show sample data
print("\nSample PPS Records:")
spark.sql("""
SELECT 
    PPS,
    KeyID,
    ClientName,
    Payor_Type,
    EarnedRev,
    Cash,
    AgingDays
FROM Report_MonthEndClose_PPS
ORDER BY EarnedRev DESC
LIMIT 10
""").show(truncate=False)

print("\n✓ Ready for PDGM and Non-PPS data retrieval")

In [0]:
# =============================================
# SECTION 1: PPS Credits Base View
# =============================================
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW v_hard_close_pps_credits AS
SELECT 
    hcc.hcc_closing_period,
    hcc.hcc_reporting_period,
    hcc.hcc_branchcode,
    hcc.hcc_paid,
    pa.pa_lastname,
    pa.pa_firstname,
    pa.pa_mi,
    pa.pa_MedicareNum,
    hcc.hcc_slid,
    sl.sl_desc,
    hcc.hcc_psid,
    ps.ps_desc,
    hcc.hcc_ptid,
    pt.pt_desc,
    hcc.hcc_fcid,
    fc.fc_desc,
    hcc.hcc_agencyid,
    agency.agency_name,
    agency.agency_ProviderNumber,
    hcc.hcc_hsid,
    hcc.hcc_isoffset,
    hcc.hcc_rfid,
    hcc.hcc_type,
    hcc.hcc_amount
FROM {hchb_hard_close_credits} hcc
LEFT JOIN {hchb_clients_all} pa ON pa.pa_id = hcc.hcc_paid
LEFT JOIN {hchb_payor_sources} ps ON ps.ps_id = hcc.hcc_psid
LEFT JOIN {hchb_payor_types} pt ON pt.pt_id = hcc.hcc_ptid
LEFT JOIN {hchb_financial_class} fc ON fc.fc_id = hcc.hcc_fcid
JOIN {hchb_agencies} agency ON agency.agency_id = hcc.hcc_agencyid
JOIN {hchb_service_lines} sl ON sl.sl_id = hcc.hcc_slid
""")

print("✓ PPS credits base view created")

# =============================================
# SECTION 2: PPS Credits Aggregated Function View
# =============================================

print("\nSection 2: Creating PPS credits aggregated view...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW fn_GetHardCloseCreditsAR AS
SELECT
    hcc_reporting_period AS reporting_period,
    hcc_branchcode AS closing_branchcode,
    hcc_paid AS paid,
    pa_lastname AS lastname,
    pa_firstname AS firstname,
    pa_mi AS mi,
    CONCAT(COALESCE(pa_lastname, ''), ', ', COALESCE(pa_firstname, ''), ' ', COALESCE(pa_mi, '')) AS fullname,
    pa_medicarenum AS medicarenum,
    hcc_slid AS slid,
    sl_desc AS sldesc,
    hcc_psid AS psid,
    ps_desc AS psdesc,
    hcc_ptid AS ptid,
    pt_desc AS ptdesc,
    hcc_fcid AS fcid,
    fc_desc AS fcdesc,
    hcc_agencyid AS agencyid,
    agency_name AS agencyname,
    agency_providernumber AS providernumber,
    hcc_type AS CreditType,
    SUM(CASE WHEN hcc_rfid IS NOT NULL THEN hcc_amount ELSE 0 END) AS CreditRefunds,
    SUM(CASE WHEN hcc_isoffset = True THEN hcc_amount ELSE 0 END) AS CreditOffsets,
    SUM(CASE WHEN hcc_rfid IS NULL AND hcc_isoffset = False THEN hcc_amount ELSE 0 END) AS Credits
FROM v_hard_close_pps_credits
WHERE hcc_closing_period <= {closing_period}
GROUP BY 
    hcc_reporting_period, hcc_branchcode, hcc_paid, pa_lastname, pa_firstname, pa_mi, pa_medicarenum,
    hcc_slid, sl_desc, hcc_psid, ps_desc, hcc_ptid, pt_desc, hcc_agencyid, agency_name, 
    agency_providernumber, hcc_type, hcc_fcid, fc_desc
""")

credits_count = spark.sql("SELECT COUNT(*) as count FROM fn_GetHardCloseCreditsAR").collect()[0]['count']
print(f"✓ PPS credits aggregated: {credits_count} records")

# =============================================
# SECTION 3: PPS Credits and Refunds Report
# =============================================

print("\nSection 3: Creating PPS credits and refunds report...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_MonthEndClose_PPS_Credits AS
SELECT
    1 AS PPS,
    0 AS KeyID,
    CAST(NULL AS DATE) AS KeyDate,
    CAST(NULL AS DATE) AS EndDate,
    f.paid AS PAID,
    CASE 
        WHEN f.paid = 0 THEN CONCAT('CREDIT', COALESCE(CONCAT(' - ', f.psdesc), ''))
        ELSE CONCAT(f.LastName, ', ', f.FirstName)
    END AS ClientName,
    f.fcid AS FCID,
    f.fcdesc AS Financial_Class,
    f.ptid AS PTID,
    f.ptdesc AS Payor_Type,
    f.psid AS PSID,
    f.psdesc AS Payor_Source,
    0 AS MSP,
    f.slid AS SLID,
    f.sldesc AS slDesc,
    f.AgencyId,
    CONCAT(f.AgencyName, ':', f.ProviderNumber) AS Agency,
    CAST(NULL AS STRING) AS InsuredId,
    CAST(NULL AS STRING) AS EOEType,
    CAST(NULL AS DATE) AS BillDate,
    f.Closing_BranchCode AS Reporting_BranchCode,
    0 AS EarnedRev,
    0 AS Adjustments,
    0 AS UnEarnedRev,
    SUM(f.Credits + f.CreditRefunds + f.creditOffsets) AS Cash,
    SUM(f.Credits) AS Credits,
    SUM(f.CreditRefunds * -1) AS Refunds,
    0 AS DaysDelayed,
    '{period_end_date}' AS LastDayClosingPeriod,
    -1 AS AgingDays,
    CAST(NULL AS BIGINT) AS RevenueDetailID,
    0 AS TCID
FROM fn_GetHardCloseCreditsAR f
JOIN Branches_Filter b ON b.BranchCode = f.Closing_BranchCode
JOIN ServiceLines_Filter sl ON sl.ID = f.slid
JOIN Agencies_Filter ag ON ag.agid = f.AgencyId
JOIN PayorTypes_Filter pt ON pt.ID = f.ptid
JOIN PayorSources_Filter ps ON ps.ID = f.psid
JOIN FinancialClasses_Filter fc ON fc.ID = f.fcid
GROUP BY 
    f.paid, f.LastName, f.FirstName, f.fcid, f.fcdesc, f.ptid, f.ptdesc, 
    f.psid, f.psdesc, f.slid, f.sldesc, f.AgencyId, f.AgencyName, 
    f.ProviderNumber, f.Closing_BranchCode
""")

pps_credits_report_count = spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PPS_Credits").collect()[0]['count']
print(f"✓ PPS credits report: {pps_credits_report_count} records")

# =============================================
# SECTION 4: PDGM - Get Initial Headers for Unearned Revenue
# =============================================

print("\nSection 4: Getting PDGM initial headers...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Hard_Close_Initial_PDGM_Headers AS
SELECT
    MAX(hcph.hcph_id) AS HCPHID,
    hcph.hcph_MSP AS MSP
FROM {hchb_hard_close_pdgm_headers} hcph
WHERE hcph.hcph_Closing_Period <= {closing_period}
GROUP BY hcph.hcph_PDGMperiodID, hcph.hcph_MSP, hcph.hcph_FCID
""")

pdgm_headers_count = spark.sql("SELECT COUNT(*) as count FROM Hard_Close_Initial_PDGM_Headers").collect()[0]['count']
print(f"✓ PDGM headers: {pdgm_headers_count} records")

# =============================================
# SECTION 5: PDGM Header Details
# =============================================

print("\nSection 5: Getting PDGM header details...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Hard_Close_PDGM_Header_Details AS
SELECT
    hcph.hcph_id,
    hcph.hcph_PDGMperiodID,
    hcph.hcph_PeriodStartDate,
    hcph.hcph_PeriodEndDate,
    hcph.hcph_Closing_Period,
    hcph.hcph_Closing_Branchcode,
    cefs.cefs_medicareNo AS hcph_InsuredId,
    hcph.hcph_PAID,
    hcph.hcph_PTID,
    hcph.hcph_MSP,
    hcph.hcph_SLID,
    hcph.hcph_FCID,
    hcph.hcph_PSID,
    hcph.hcph_Agencyid,
    CASE 
        WHEN hcph.hcph_IsPEP = True THEN 'P'
        WHEN hcph.hcph_ReimbursementType IN ('L', 'O') THEN hcph.hcph_ReimbursementType
        WHEN hcph.hcph_ReimbursementType = 'S' THEN NULL
        ELSE NULL 
    END AS hcph_EOEType,
    hcph.hcph_BillDate,
    hcph.hcph_epiid,
    hcph.hcph_AdmitDate,
    hcph.hcph_DischargeDate
FROM {hchb_hard_close_pdgm_headers} hcph
JOIN Hard_Close_Initial_PDGM_Headers hcih 
    ON hcih.HCPHID = hcph.hcph_id AND hcih.MSP = hcph.hcph_MSP
JOIN {hchb_pdgm_period} pp ON pp.pp_id = hcph.hcph_PDGMperiodID
JOIN {hchb_client_episode_fs} cefs ON cefs.cefs_id = pp.pp_cefsid
""")

pdgm_details_count = spark.sql("SELECT COUNT(*) as count FROM Hard_Close_PDGM_Header_Details").collect()[0]['count']
print(f"✓ PDGM header details: {pdgm_details_count} records")

# =============================================
# SECTION 6: PDGM Earned/Unearned Revenue and Adjustments
# =============================================

print("\nSection 6: Calculating PDGM revenue and adjustments...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ReportDetailsPDGM_Revenue AS
SELECT
    hcph.hcph_PDGMperiodID AS KeyID,
    hcph.hcph_FCID AS FCID,
    hcph.hcph_PTID AS PTID,
    hcph.hcph_PSID AS PSID,
    hcph.hcph_MSP AS MSP,
    hcph.hcph_SLID AS SLID,
    hcph.hcph_Agencyid AS AgencyId,
    hcpr.hcpr_Reporting_Branchcode AS Reporting_Branchcode,
    SUM(CASE WHEN hcpr.hcpr_Type = 'E' THEN hcpr.hcpr_Amount ELSE 0 END) AS EarnedRev,
    SUM(CASE 
        WHEN hcpr.hcpr_Type = 'A' THEN hcpr.hcpr_Amount
        WHEN hcpr.hcpr_Type = 'M' AND hcpr.hcpr_Reporting_Period <= {closing_period} THEN hcpr.hcpr_Amount
        ELSE 0 
    END) AS Adjustments,
    SUM(CASE 
        WHEN hcpr.hcpr_Type = 'U' THEN hcpr.hcpr_Amount
        WHEN hcpr.hcpr_Type = 'M' AND hcpr.hcpr_Reporting_Period > {closing_period} THEN hcpr.hcpr_Amount
        ELSE 0 
    END) AS UnEarnedRev,
    0 AS Cash,
    0 AS Credits,
    0 AS Refunds
FROM {hchb_hard_close_pdgm_revenue} hcpr
JOIN {hchb_hard_close_pdgm_headers} hcph ON hcpr.hcpr_hcphid = hcph.hcph_id
JOIN Branches_Filter b ON b.BranchCode = hcpr.hcpr_Reporting_Branchcode
JOIN ServiceLines_Filter slp ON slp.ID = hcph.hcph_slid
JOIN Agencies_Filter agp ON agp.agid = hcph.hcph_Agencyid
JOIN PayorTypes_Filter ptp ON ptp.ID = hcph.hcph_ptid
JOIN PayorSources_Filter psp ON psp.ID = hcph.hcph_psid
JOIN FinancialClasses_Filter fcp ON fcp.ID = hcph.hcph_fcid
WHERE hcph.hcph_Closing_Period <= {closing_period}
  AND (
      hcph.hcph_PeriodStartDate <= '{max_period_date}'
      OR '{pdgm_show_future_period_unearned}' = 'Y'
  )
  AND (
      hcpr.hcpr_Type <> 'U'
      OR hcph.hcph_Closing_Period = {closing_period}
  )
GROUP BY
    hcph.hcph_PDGMperiodID, hcph.hcph_FCID, hcph.hcph_PTID, hcph.hcph_PSID,
    hcph.hcph_MSP, hcph.hcph_SLID, hcph.hcph_Agencyid, hcpr.hcpr_Reporting_Branchcode
""")

pdgm_revenue_count = spark.sql("SELECT COUNT(*) as count FROM ReportDetailsPDGM_Revenue").collect()[0]['count']
print(f"✓ PDGM revenue records: {pdgm_revenue_count}")

# =============================================
# SECTION 7: PDGM Cash
# =============================================

print("\nSection 7: Processing PDGM cash receipts...")

# Handle NULL cash_received_through
cash_condition = f"""
(
    {cash_received_through} IS NULL
    OR ({cash_received_through} IS NOT NULL AND hcph.hcph_Closing_Period <= {cash_received_through})
)
""" if cash_received_through is not None else "1=1"

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ReportDetailsPDGM_Cash AS
SELECT
    hcph.hcph_PDGMperiodID AS KeyID,
    hcph.hcph_FCID AS FCID,
    hcph.hcph_PTID AS PTID,
    hcph.hcph_PSID AS PSID,
    hcph.hcph_MSP AS MSP,
    hcph.hcph_SLID AS SLID,
    hcph.hcph_Agencyid AS AgencyId,
    hcpc.hcpc_Reporting_Branchcode AS Reporting_Branchcode,
    0 AS EarnedRev,
    0 AS Adjustments,
    0 AS UnEarnedRev,
    SUM(hcpc.hcpc_Amount) AS Cash,
    0 AS Credits,
    0 AS Refunds
FROM {hchb_hard_close_pdgm_cash} hcpc
JOIN {hchb_hard_close_pdgm_headers} hcph ON hcpc.hcpc_hcphid = hcph.hcph_id
JOIN Branches_Filter b ON b.BranchCode = hcpc.hcpc_Reporting_Branchcode
JOIN ServiceLines_Filter slp ON slp.ID = hcph.hcph_slid
JOIN Agencies_Filter agp ON agp.agid = hcph.hcph_Agencyid
JOIN PayorTypes_Filter ptp ON ptp.ID = hcph.hcph_ptid
JOIN PayorSources_Filter psp ON psp.ID = hcph.hcph_psid
JOIN FinancialClasses_Filter fcp ON fcp.ID = hcph.hcph_fcid
WHERE hcph.hcph_Closing_Period <= {closing_period}
  AND {cash_condition}
  AND hcpc.hcpc_Type <> 'C'
GROUP BY
    hcph.hcph_PDGMperiodID, hcph.hcph_FCID, hcph.hcph_PTID, hcph.hcph_PSID,
    hcph.hcph_MSP, hcph.hcph_SLID, hcph.hcph_Agencyid, hcpc.hcpc_Reporting_Branchcode
""")

pdgm_cash_count = spark.sql("SELECT COUNT(*) as count FROM ReportDetailsPDGM_Cash").collect()[0]['count']
print(f"✓ PDGM cash records: {pdgm_cash_count}")

# =============================================
# SECTION 8: Combine PDGM Details
# =============================================

print("\nSection 8: Combining PDGM details...")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW ReportDetailsPDGM_Combined AS
SELECT
    KeyID, FCID, PTID, PSID, MSP, SLID, AgencyId, Reporting_Branchcode,
    SUM(EarnedRev) AS EarnedRev,
    SUM(Adjustments) AS Adjustments,
    SUM(UnEarnedRev) AS UnEarnedRev,
    SUM(Cash) AS Cash,
    SUM(Credits) AS Credits,
    SUM(Refunds) AS Refunds
FROM (
    SELECT * FROM ReportDetailsPDGM_Revenue
    UNION ALL
    SELECT * FROM ReportDetailsPDGM_Cash
)
GROUP BY KeyID, FCID, PTID, PSID, MSP, SLID, AgencyId, Reporting_Branchcode
""")

pdgm_combined_count = spark.sql("SELECT COUNT(*) as count FROM ReportDetailsPDGM_Combined").collect()[0]['count']
print(f"✓ PDGM combined records: {pdgm_combined_count}")

# =============================================
# SUMMARY
# =============================================

print("\n" + "=" * 60)
print("PART 3 COMPLETED - PPS CREDITS AND PDGM DATA")
print("=" * 60)
print(f"PPS Credits: {pps_credits_report_count}")
print(f"PDGM Combined: {pdgm_combined_count}")
print("\n✓ Ready for Part 4: PDGM Credits/Final Report and Non-PPS Data")

In [0]:
# =============================================
# SECTION 1: PDGM Credits and Refunds
# =============================================
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_MonthEndClose_PDGM_Credits AS
SELECT
    2 AS PPS,
    0 AS KeyID,
    CAST(NULL AS DATE) AS KeyDate,
    CAST(NULL AS DATE) AS EndDate,
    hcpc.hcpc_paid AS PAID,
    CASE 
        WHEN hcpc.hcpc_paid = 0 THEN CONCAT('CREDIT', COALESCE(CONCAT(' - ', ps.ps_Desc), ''))
        ELSE CONCAT(pa.pa_LastName, ', ', pa.pa_FirstName)
    END AS ClientName,
    hcpc.hcpc_FCID AS FCID,
    fc.fc_Desc AS Financial_Class,
    hcpc.hcpc_PTID AS PTID,
    pt.pt_Desc AS Payor_Type,
    hcpc.hcpc_psid AS PSID,
    ps.ps_Desc AS Payor_Source,
    0 AS MSP,
    hcpc.hcpc_SLID AS SLID,
    sl.sl_Desc AS slDesc,
    hcpc.hcpc_Agencyid AS AgencyId,
    CONCAT(Agency.Agency_Name, ':', Agency.Agency_ProviderNumber) AS Agency,
    CAST(NULL AS STRING) AS InsuredId,
    CAST(NULL AS STRING) AS EOEType,
    CAST(NULL AS DATE) AS BillDate,
    hcpc.hcpc_Branchcode AS Reporting_BranchCode,
    0 AS EarnedRev,
    0 AS Adjustments,
    0 AS UnEarnedRev,
    SUM(CASE 
        WHEN (hcpc.hcpc_rfid IS NULL AND hcpc.hcpc_isoffset = False)
             OR hcpc.hcpc_isoffset = True
             OR hcpc.hcpc_rfid IS NOT NULL
        THEN hcpc.hcpc_amount 
        ELSE 0 
    END) AS Cash,
    SUM(CASE WHEN hcpc.hcpc_rfid IS NULL AND hcpc.hcpc_isoffset = False THEN hcpc.hcpc_amount ELSE 0 END) AS Credits,
    SUM(CASE WHEN hcpc.hcpc_rfid IS NOT NULL THEN hcpc.hcpc_amount ELSE 0 END) AS Refunds,
    0 AS DaysDelayed,
    '{period_end_date}' AS LastDayClosingPeriod,
    -1 AS AgingDays,
    CAST(NULL AS BIGINT) AS RevenueDetailID,
    0 AS TCID
FROM {hchb_hard_close_pdgm_credits} hcpc
JOIN Branches_Filter b ON b.BranchCode = hcpc.hcpc_BranchCode
JOIN ServiceLines_Filter slp ON slp.ID = hcpc.hcpc_slid
JOIN Agencies_Filter agp ON agp.agid = hcpc.hcpc_Agencyid
JOIN PayorTypes_Filter ptp ON ptp.ID = hcpc.hcpc_ptid
JOIN PayorSources_Filter psp ON psp.ID = hcpc.hcpc_psid
JOIN FinancialClasses_Filter fcp ON fcp.ID = hcpc.hcpc_fcid
LEFT JOIN {hchb_financial_class} fc ON fc.fc_id = hcpc.hcpc_fcid
LEFT JOIN {hchb_clients_all} pa ON pa.pa_id = hcpc.hcpc_PAID
LEFT JOIN {hchb_agencies} Agency ON Agency.Agency_id = hcpc.hcpc_Agencyid
LEFT JOIN {hchb_service_lines} sl ON sl.sl_id = hcpc.hcpc_SLID
LEFT JOIN {hchb_payor_types} pt ON pt.pt_id = hcpc.hcpc_PTID
LEFT JOIN {hchb_payor_sources} ps ON ps.ps_id = hcpc.hcpc_PSID
WHERE hcpc.hcpc_Closing_Period <= {closing_period}
GROUP BY 
    hcpc.hcpc_paid, pa.pa_LastName, pa.pa_FirstName, hcpc.hcpc_FCID, fc.fc_Desc,
    hcpc.hcpc_PTID, pt.pt_Desc, hcpc.hcpc_psid, ps.ps_Desc, hcpc.hcpc_SLID, sl.sl_Desc,
    hcpc.hcpc_Agencyid, Agency.Agency_Name, Agency.Agency_ProviderNumber, hcpc.hcpc_Branchcode
""")

pdgm_credits_count = spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PDGM_Credits").collect()[0]['count']
print(f"✓ PDGM credits report: {pdgm_credits_count} records")

# =============================================
# SECTION 2: PDGM Final Output
# =============================================

print("\nSection 2: Creating PDGM final output report...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_MonthEndClose_PDGM AS
SELECT
    2 AS PPS,
    r.KeyID,
    hcph.hcph_PeriodStartDate AS KeyDate,
    hcph.hcph_PeriodEndDate AS EndDate,
    hcph.hcph_paid AS PAID,
    CONCAT(COALESCE(pa.pa_LastName, ''), CASE WHEN pa.pa_LastName IS NOT NULL THEN ', ' ELSE '' END, COALESCE(pa.pa_FirstName, '')) AS ClientName,
    r.FCID,
    fc.fc_Desc AS Financial_Class,
    r.PTID,
    CONCAT(pt.pt_Desc, CASE WHEN hcph.hcph_MSP = True THEN ' (MSP)' ELSE '' END) AS Payor_Type,
    r.PSID,
    ps.ps_Desc AS Payor_Source,
    r.MSP,
    r.SLID,
    sl.sl_Desc AS slDesc,
    r.AgencyId,
    CONCAT(Agency.Agency_Name, ':', Agency.Agency_ProviderNumber) AS Agency,
    hcph.hcph_InsuredId AS InsuredId,
    hcph.hcph_EOEType AS EOEType,
    hcph.hcph_BillDate AS BillDate,
    r.Reporting_Branchcode AS Reporting_BranchCode,
    SUM(COALESCE(r.EarnedRev, 0)) AS EarnedRev,
    SUM(COALESCE(r.Adjustments, 0)) AS Adjustments,
    SUM(COALESCE(r.UnEarnedRev, 0)) AS UnEarnedRev,
    SUM(COALESCE(r.Cash, 0)) AS Cash,
    SUM(COALESCE(r.Credits, 0)) AS Credits,
    SUM(COALESCE(r.Refunds * -1, 0)) AS Refunds,
    MAX(CASE 
        WHEN DATEDIFF(hcph.hcph_BillDate, hcph.hcph_PeriodEndDate) < 0 THEN 0 
        ELSE COALESCE(DATEDIFF(hcph.hcph_BillDate, hcph.hcph_PeriodEndDate), 0) 
    END) AS DaysDelayed,
    '{period_end_date}' AS LastDayClosingPeriod,
    MAX(COALESCE(
        CASE 
            WHEN {age_from_soe} = 1 THEN DATEDIFF('{period_end_date}', hcph.hcph_PeriodStartDate)
            ELSE DATEDIFF('{period_end_date}', hcph.hcph_PeriodEndDate)
        END, -1
    )) AS AgingDays,
    CAST(NULL AS BIGINT) AS RevenueDetailID,
    0 AS TCID
FROM ReportDetailsPDGM_Combined r
JOIN Hard_Close_PDGM_Header_Details hcph 
    ON hcph.hcph_PDGMperiodID = r.KeyID 
    AND hcph.hcph_MSP = r.MSP
    AND hcph.hcph_FCID = r.FCID
LEFT JOIN {hchb_financial_class} fc ON fc.fc_id = hcph.hcph_fcid
LEFT JOIN {hchb_clients_all} pa ON pa.pa_id = hcph.hcph_PAID
LEFT JOIN {hchb_agencies} Agency ON Agency.Agency_id = hcph.hcph_Agencyid
LEFT JOIN {hchb_service_lines} sl ON sl.sl_id = hcph.hcph_SLID
LEFT JOIN {hchb_payor_types} pt ON pt.pt_id = hcph.hcph_PTID
LEFT JOIN {hchb_payor_sources} ps ON ps.ps_id = hcph.hcph_PSID
GROUP BY 
    r.KeyID, hcph.hcph_PeriodStartDate, hcph.hcph_PeriodEndDate, hcph.hcph_paid,
    pa.pa_LastName, pa.pa_FirstName, r.FCID, fc.fc_Desc, r.PTID, pt.pt_Desc, hcph.hcph_MSP,
    r.PSID, ps.ps_Desc, r.MSP, r.SLID, sl.sl_Desc, r.AgencyId, Agency.Agency_Name, 
    Agency.Agency_ProviderNumber, hcph.hcph_InsuredId, hcph.hcph_EOEType, 
    hcph.hcph_BillDate, r.Reporting_Branchcode
""")

pdgm_final_count = spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PDGM").collect()[0]['count']
print(f"✓ PDGM final report: {pdgm_final_count} records")

# =============================================
# SECTION 3: Non-PPS - Most Recent Headers
# =============================================

print("\nSection 3: Processing Non-PPS most recent headers...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW NonPPS_MostRecentHeader AS
SELECT
    hcnbs_paid AS paid,
    hcnbs_BranchCode AS closing_branchcode,
    hcnbs_psid AS psid,
    hcnbs_AgencyId AS agencyid,
    hcnbs_hcnhid AS hcnhid,
    ROW_NUMBER() OVER (
        PARTITION BY hcnbs_paid, hcnbs_BranchCode, hcnbs_psid, hcnbs_AgencyId 
        ORDER BY hcnbs_Closing_Period DESC
    ) AS rownum
FROM {hchb_hard_close_nonpps_balance_snapshot}
WHERE hcnbs_Closing_Period <= {closing_period}
  AND hcnbs_bucketChange = False
  AND hcnbs_hcnhid IS NOT NULL
""")

nonpps_header_count = spark.sql("SELECT COUNT(*) as count FROM NonPPS_MostRecentHeader").collect()[0]['count']
print(f"✓ Non-PPS most recent headers: {nonpps_header_count} records")

# Handle NULL cash_received_through for second part of UNION
cash_received_through_value = cash_received_through if cash_received_through is not None else 'NULL'

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW HCNH_for_HCNBS AS
SELECT
    mrh.hcnhid AS hcnh_id,
    hcnbs.hcnbs_id AS hcnbs_id
FROM {hchb_hard_close_nonpps_balance_snapshot} hcnbs
JOIN NonPPS_MostRecentHeader mrh 
    ON hcnbs.hcnbs_paid = mrh.paid
    AND hcnbs.hcnbs_BranchCode = mrh.closing_branchcode
    AND hcnbs.hcnbs_psid = mrh.psid
    AND hcnbs.hcnbs_AgencyId = mrh.agencyid
WHERE hcnbs.hcnbs_Closing_Period = {closing_period}
  AND hcnbs.hcnbs_bucketChange = False
  AND mrh.rownum = 1

UNION ALL

SELECT
    mrh.hcnhid AS hcnh_id,
    hcnbs.hcnbs_id AS hcnbs_id
FROM {hchb_hard_close_nonpps_balance_snapshot} hcnbs
JOIN NonPPS_MostRecentHeader mrh 
    ON hcnbs.hcnbs_paid = mrh.paid
    AND hcnbs.hcnbs_BranchCode = mrh.closing_branchcode
    AND hcnbs.hcnbs_psid = mrh.psid
    AND hcnbs.hcnbs_AgencyId = mrh.agencyid
WHERE hcnbs.hcnbs_Closing_Period > {closing_period}
  AND hcnbs.hcnbs_Closing_Period <= {cash_received_through_value}
  AND hcnbs.hcnbs_bucketChange = False
  AND mrh.rownum = 1
  AND {cash_received_through_value} IS NOT NULL
""")

hcnh_mapping_count = spark.sql("SELECT COUNT(*) as count FROM HCNH_for_HCNBS").collect()[0]['count']
print(f"✓ HCNH mapping records: {hcnh_mapping_count}")

# =============================================
# SUMMARY
# =============================================

print("\n" + "=" * 60)
print("PART 4 COMPLETED - PDGM FINAL AND NON-PPS")
print("=" * 60)

# Show summary of all reports created so far
summary_data = [
    ("PPS Report", spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PPS").collect()[0]['count']),
    ("PPS Credits", spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PPS_Credits").collect()[0]['count']),
    ("PDGM Report", pdgm_final_count),
    ("PDGM Credits", pdgm_credits_count),
    ("Non-PPS Headers", nonpps_header_count),
]

print("\nReport Summary:")
for report_name, count in summary_data:
    print(f"  {report_name}: {count} records")

print("\n✓ Ready for Part 5: Non-PPS Final Report and Aggregations")

In [0]:
# =============================================
# SECTION 1: Non-PPS Current Period Data
# =============================================
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_MonthEndClose_NonPPS_Current AS
SELECT
    0 AS PPS,
    COALESCE(hcnbs.hcnbs_paid, 0) AS KeyID,
    0 AS TCID,
    CAST(NULL AS DATE) AS KeyDate,
    CAST(NULL AS DATE) AS EndDate,
    COALESCE(hcnbs.hcnbs_paid, 0) AS PAID,
    CASE 
        WHEN hcnbsd.hcnbsd_DateOfService IS NULL AND hcnbs.hcnbs_paid IS NULL 
            THEN CONCAT('CREDIT', COALESCE(CONCAT(' - ', ps.ps_desc), ''))
        WHEN hcnh.hcnh_id IS NOT NULL 
            THEN CONCAT(COALESCE(hcnh.hcnh_LastName, ''), COALESCE(CONCAT(', ', hcnh.hcnh_FirstName), ''))
        WHEN Clients.pa_id IS NOT NULL 
            THEN CONCAT(COALESCE(clients.pa_LastName, ''), COALESCE(CONCAT(', ', clients.pa_FirstName), ''))
        ELSE 'CREDIT' 
    END AS ClientName,
    hcnbs.hcnbs_fcid AS FCID,
    fc.fc_desc AS Financial_Class,
    hcnbs.hcnbs_ptid AS PTID,
    pt.pt_desc AS Payor_Type,
    hcnbs.hcnbs_psid AS PSID,
    ps.ps_desc AS Payor_Source,
    CAST(false AS BOOLEAN) AS MSP,
    hcnbs.hcnbs_slid AS SLID,
    sl.sl_desc AS slDesc,
    hcnbs.hcnbs_AgencyId AS AgencyId,
    CONCAT(ag.agency_name, ':', ag.agency_ProviderNumber) AS Agency,
    CAST(NULL AS STRING) AS InsuredId,
    CAST(NULL AS STRING) AS EOEType,
    CASE WHEN hcnbsd.hcnbsd_DateOfService IS NULL THEN NULL ELSE hcnh.hcnh_BillDate END AS BillDate,
    hcnbs.hcnbs_BranchCode AS Reporting_BranchCode,
    hcnbsd.hcnbsd_RevenueClosingBalance AS EarnedRev,
    0 AS Adjustments,
    0 AS UnEarnedRev,
    hcnbsd.hcnbsd_CashClosingBalance AS Cash,
    CASE WHEN hcnbsd.hcnbsd_DateOfService IS NULL THEN hcnbsd.hcnbsd_CashClosingBalance ELSE 0 END AS Credits,
    0 AS Refunds,
    0 AS DaysDelayed,
    '{period_end_date}' AS LastDayClosingPeriod,
    COALESCE(DATEDIFF('{period_end_date}', hcnbsd.hcnbsd_DateOfService), -1) AS AgingDays,
    CAST(NULL AS BIGINT) AS RevenueDetailID
FROM {hchb_hard_close_nonpps_balance_snapshot} hcnbs
JOIN {hchb_hard_close_nonpps_balance_snapshot_days} hcnbsd 
    ON hcnbs.hcnbs_id = hcnbsd.hcnbsd_hcnbsid
JOIN Branches_Filter bm ON bm.BranchCode = hcnbs.hcnbs_BranchCode
JOIN ServiceLines_Filter slm ON slm.ID = hcnbs.hcnbs_slid
JOIN Agencies_Filter agm ON agm.agid = hcnbs.hcnbs_AgencyId
JOIN PayorTypes_Filter ptm ON ptm.ID = hcnbs.hcnbs_ptid
JOIN PayorSources_Filter psm ON psm.ID = hcnbs.hcnbs_psid
JOIN FinancialClasses_Filter fcm ON fcm.ID = hcnbs.hcnbs_fcid
JOIN {hchb_financial_class} fc ON fc.fc_id = hcnbs.hcnbs_fcid
JOIN {hchb_payor_types} pt ON pt.pt_id = hcnbs.hcnbs_ptid
JOIN {hchb_payor_sources} ps ON ps.ps_id = hcnbs.hcnbs_psid
JOIN {hchb_service_lines} sl ON sl.sl_id = hcnbs.hcnbs_slid
JOIN {hchb_agencies} ag ON ag.agency_id = hcnbs.hcnbs_AgencyId
LEFT JOIN HCNH_for_HCNBS hfh ON hcnbs.hcnbs_id = hfh.hcnbs_id
LEFT JOIN {hchb_hard_close_nonpps_headers} hcnh ON hcnh.hcnh_id = hfh.hcnh_id
LEFT JOIN {hchb_clients_all} clients ON clients.pa_id = hcnbs.hcnbs_paid
WHERE hcnbs.hcnbs_Closing_Period = {closing_period}
""")

nonpps_current_count = spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_NonPPS_Current").collect()[0]['count']
print(f"✓ Non-PPS current period: {nonpps_current_count} records")

# =============================================
# SECTION 2: Non-PPS Unearned Revenue
# =============================================

print("\nSection 2: Creating Non-PPS unearned revenue...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_MonthEndClose_NonPPS_Unearned AS
SELECT
    0 AS PPS,
    COALESCE(hcnbs.hcnbs_paid, 0) AS KeyID,
    0 AS TCID,
    CAST(NULL AS DATE) AS KeyDate,
    CAST(NULL AS DATE) AS EndDate,
    COALESCE(hcnbs.hcnbs_paid, 0) AS PAID,
    CASE 
        WHEN hcnbs.hcnbs_paid IS NULL THEN CONCAT('CREDIT', COALESCE(CONCAT(' - ', ps.ps_desc), ''))
        WHEN hcnh.hcnh_id IS NOT NULL THEN CONCAT(COALESCE(hcnh.hcnh_LastName, ''), COALESCE(CONCAT(', ', hcnh.hcnh_FirstName), ''))
        WHEN clients.pa_id IS NOT NULL THEN CONCAT(COALESCE(clients.pa_LastName, ''), COALESCE(CONCAT(', ', clients.pa_FirstName), ''))
        ELSE 'CREDIT' 
    END AS ClientName,
    hcnbs.hcnbs_fcid AS FCID,
    fc.fc_desc AS Financial_Class,
    hcnbs.hcnbs_ptid AS PTID,
    pt.pt_desc AS Payor_Type,
    hcnbs.hcnbs_psid AS PSID,
    ps.ps_desc AS Payor_Source,
    CAST(false AS BOOLEAN) AS MSP,
    hcnbs.hcnbs_slid AS SLID,
    sl.sl_desc AS slDesc,
    hcnbs.hcnbs_AgencyId AS AgencyId,
    CONCAT(ag.agency_name, ':', ag.agency_ProviderNumber) AS Agency,
    CAST(NULL AS STRING) AS InsuredId,
    CAST(NULL AS STRING) AS EOEType,
    CASE WHEN hcnh.hcnh_BillDate IS NULL THEN NULL ELSE hcnh.hcnh_BillDate END AS BillDate,
    hcnbs.hcnbs_BranchCode AS Reporting_BranchCode,
    0 AS EarnedRev,
    0 AS Adjustments,
    hcnbs.hcnbs_unEarnedrevenue AS UnEarnedRev,
    0 AS Cash,
    0 AS Credits,
    0 AS Refunds,
    0 AS DaysDelayed,
    '{period_end_date}' AS LastDayClosingPeriod,
    -1 AS AgingDays,
    CAST(NULL AS BIGINT) AS RevenueDetailID
FROM {hchb_hard_close_nonpps_balance_snapshot} hcnbs
JOIN Branches_Filter bm ON bm.BranchCode = hcnbs.hcnbs_BranchCode
JOIN ServiceLines_Filter slm ON slm.ID = hcnbs.hcnbs_slid
JOIN Agencies_Filter agm ON agm.agid = hcnbs.hcnbs_AgencyId
JOIN PayorTypes_Filter ptm ON ptm.ID = hcnbs.hcnbs_ptid
JOIN PayorSources_Filter psm ON psm.ID = hcnbs.hcnbs_psid
JOIN FinancialClasses_Filter fcm ON fcm.ID = hcnbs.hcnbs_fcid
JOIN {hchb_financial_class} fc ON fc.fc_id = hcnbs.hcnbs_fcid
JOIN {hchb_payor_types} pt ON pt.pt_id = hcnbs.hcnbs_ptid
JOIN {hchb_payor_sources} ps ON ps.ps_id = hcnbs.hcnbs_psid
JOIN {hchb_service_lines} sl ON sl.sl_id = hcnbs.hcnbs_slid
JOIN {hchb_agencies} ag ON ag.agency_id = hcnbs.hcnbs_AgencyId
LEFT JOIN HCNH_for_HCNBS hfh ON hcnbs.hcnbs_id = hfh.hcnbs_id
LEFT JOIN {hchb_hard_close_nonpps_headers} hcnh ON hcnh.hcnh_id = hfh.hcnh_id
LEFT JOIN {hchb_clients_all} clients ON clients.pa_id = hcnbs.hcnbs_paid
WHERE hcnbs.hcnbs_Closing_Period = {closing_period}
  AND hcnbs.hcnbs_unEarnedrevenue <> 0
""")

nonpps_unearned_count = spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_NonPPS_Unearned").collect()[0]['count']
print(f"✓ Non-PPS unearned revenue: {nonpps_unearned_count} records")

# =============================================
# SECTION 3: Combine All Report Data
# =============================================

print("\nSection 3: Combining all report data...")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW Report_MonthEndClose_Combined AS
SELECT 
    PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class,
    PTID, Payor_Type, PSID, Payor_Source, 
    CAST(MSP AS BOOLEAN) AS MSP,
    SLID, slDesc, AgencyId, Agency, InsuredId, EOEType, BillDate, 
    Reporting_BranchCode, EarnedRev, Adjustments, UnEarnedRev, Cash, Credits, 
    Refunds, DaysDelayed, LastDayClosingPeriod, AgingDays, RevenueDetailID
FROM Report_MonthEndClose_PPS

UNION ALL

SELECT 
    PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class,
    PTID, Payor_Type, PSID, Payor_Source, 
    CAST(MSP AS BOOLEAN) AS MSP,
    SLID, slDesc, AgencyId, Agency, InsuredId, EOEType, BillDate, 
    Reporting_BranchCode, EarnedRev, Adjustments, UnEarnedRev, Cash, Credits, 
    Refunds, DaysDelayed, LastDayClosingPeriod, AgingDays, RevenueDetailID
FROM Report_MonthEndClose_PPS_Credits

UNION ALL

SELECT 
    PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class,
    PTID, Payor_Type, PSID, Payor_Source, 
    CAST(MSP AS BOOLEAN) AS MSP,
    SLID, slDesc, AgencyId, Agency, InsuredId, EOEType, BillDate, 
    Reporting_BranchCode, EarnedRev, Adjustments, UnEarnedRev, Cash, Credits, 
    Refunds, DaysDelayed, LastDayClosingPeriod, AgingDays, RevenueDetailID
FROM Report_MonthEndClose_PDGM

UNION ALL

SELECT 
    PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class,
    PTID, Payor_Type, PSID, Payor_Source, 
    CAST(MSP AS BOOLEAN) AS MSP,
    SLID, slDesc, AgencyId, Agency, InsuredId, EOEType, BillDate, 
    Reporting_BranchCode, EarnedRev, Adjustments, UnEarnedRev, Cash, Credits, 
    Refunds, DaysDelayed, LastDayClosingPeriod, AgingDays, RevenueDetailID
FROM Report_MonthEndClose_PDGM_Credits

UNION ALL

SELECT 
    PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class,
    PTID, Payor_Type, PSID, Payor_Source, 
    CAST(MSP AS BOOLEAN) AS MSP,
    SLID, slDesc, AgencyId, Agency, InsuredId, EOEType, BillDate, 
    Reporting_BranchCode, EarnedRev, Adjustments, UnEarnedRev, Cash, Credits, 
    Refunds, DaysDelayed, LastDayClosingPeriod, AgingDays, RevenueDetailID
FROM Report_MonthEndClose_NonPPS_Current

UNION ALL

SELECT 
    PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class,
    PTID, Payor_Type, PSID, Payor_Source, 
    CAST(MSP AS BOOLEAN) AS MSP,
    SLID, slDesc, AgencyId, Agency, InsuredId, EOEType, BillDate, 
    Reporting_BranchCode, EarnedRev, Adjustments, UnEarnedRev, Cash, Credits, 
    Refunds, DaysDelayed, LastDayClosingPeriod, AgingDays, RevenueDetailID
FROM Report_MonthEndClose_NonPPS_Unearned
""")

combined_count = spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_Combined").collect()[0]['count']
print(f"✓ Combined report: {combined_count} records")

# =============================================
# SECTION 4: Line Item Report with Exclusions
# =============================================

print("\nSection 4: Creating line item report with exclusions...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_LineItem_MonthEndCloseAR AS
SELECT
    c.PPS, c.KeyID, c.TCID, c.KeyDate, c.EndDate, c.PAID, c.ClientName, c.FCID, c.Financial_Class,
    c.PTID, c.Payor_Type, c.PSID, c.Payor_Source, c.SLID, c.slDesc, c.MSP, c.Agency, c.AgencyId,
    c.InsuredId, c.EOEType, c.BillDate, c.Reporting_BranchCode,
    (COALESCE(c.EarnedRev, 0) + COALESCE(c.Adjustments, 0) + COALESCE(c.UnEarnedRev, 0)) AS Revenue,
    c.EarnedRev, c.Adjustments, c.UnEarnedRev, c.Cash, c.Credits, c.Refunds, c.DaysDelayed,
    (COALESCE(c.EarnedRev, 0) + COALESCE(c.Adjustments, 0) + COALESCE(c.UnEarnedRev, 0) - COALESCE(c.Cash, 0)) AS GrossAR,
    (COALESCE(c.EarnedRev, 0) + COALESCE(c.Adjustments, 0) - COALESCE(c.Cash, 0)) AS NetEarnedAR,
    c.LastDayClosingPeriod, c.AgingDays, c.RevenueDetailID
FROM Report_MonthEndClose_Combined c
WHERE (
    ((COALESCE(c.EarnedRev, 0) + COALESCE(c.Adjustments, 0) + COALESCE(c.UnEarnedRev, 0) - COALESCE(c.Cash, 0)) < {p_exclude_lower})
    OR 
    ((COALESCE(c.EarnedRev, 0) + COALESCE(c.Adjustments, 0) + COALESCE(c.UnEarnedRev, 0) - COALESCE(c.Cash, 0)) > {p_exclude_upper})
    OR 
    (c.Credits IS NOT NULL OR c.Refunds IS NOT NULL)
)
""")

line_item_count = spark.sql("SELECT COUNT(*) as count FROM Report_LineItem_MonthEndCloseAR").collect()[0]['count']
print(f"✓ Line item report: {line_item_count} records")

# =============================================
# SECTION 5: Group By Client and Payor Source
# =============================================

print("\nSection 5: Creating grouped by client and payor source...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_GroupByClientandPS AS
SELECT 
    a.PAID,
    a.PSID,
    SUM((COALESCE(a.EarnedRev, 0) + COALESCE(a.Adjustments, 0) + COALESCE(a.UnEarnedRev, 0) - COALESCE(a.Cash, 0))) AS AR
FROM Report_MonthEndClose_Combined a
GROUP BY a.PAID, a.PSID
HAVING 
    SUM((COALESCE(a.EarnedRev, 0) + COALESCE(a.Adjustments, 0) + COALESCE(a.UnEarnedRev, 0) - COALESCE(a.Cash, 0))) < {p_exclude_lower}
    OR 
    SUM((COALESCE(a.EarnedRev, 0) + COALESCE(a.Adjustments, 0) + COALESCE(a.UnEarnedRev, 0) - COALESCE(a.Cash, 0))) > {p_exclude_upper}
""")

grouped_ps_count = spark.sql("SELECT COUNT(*) as count FROM Report_GroupByClientandPS").collect()[0]['count']
print(f"✓ Grouped by client/PS: {grouped_ps_count} records")

# =============================================
# SUMMARY
# =============================================

print("\n" + "=" * 60)
print("PART 5 COMPLETED - NON-PPS FINAL AND AGGREGATIONS")
print("=" * 60)

# Show comprehensive summary
summary_data = [
    ("PPS Report", spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PPS").collect()[0]['count']),
    ("PPS Credits", spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PPS_Credits").collect()[0]['count']),
    ("PDGM Report", spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PDGM").collect()[0]['count']),
    ("PDGM Credits", spark.sql("SELECT COUNT(*) as count FROM Report_MonthEndClose_PDGM_Credits").collect()[0]['count']),
    ("Non-PPS Current", nonpps_current_count),
    ("Non-PPS Unearned", nonpps_unearned_count),
    ("Combined Report", combined_count),
    ("Line Item Report", line_item_count),
    ("Grouped by Client/PS", grouped_ps_count),
]

print("\nComprehensive Report Summary:")
for report_name, count in summary_data:
    print(f"  {report_name}: {count} records")

# Show financial summary
print("\nFinancial Summary:")
financial_summary = spark.sql("""
SELECT 
    SUM(COALESCE(EarnedRev, 0)) as TotalEarnedRev,
    SUM(COALESCE(Adjustments, 0)) as TotalAdjustments,
    SUM(COALESCE(UnEarnedRev, 0)) as TotalUnEarnedRev,
    SUM(COALESCE(Cash, 0)) as TotalCash,
    SUM(COALESCE(Credits, 0)) as TotalCredits,
    SUM(COALESCE(Refunds, 0)) as TotalRefunds,
    SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) as TotalGrossAR
FROM Report_MonthEndClose_Combined
""")
financial_summary.show(truncate=False)

print("\n✓ Ready for Part 6: Final Aggregations and Output")

In [0]:
# =============================================
# SECTION 1: Line Item Grouped Report
# For when BalanceDeterminedAt = 3
# =============================================
aging_filter_condition = f"r.AgingDays >= {aging_filter}" if aging_filter is not None else "1=1"

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_LineItem_Grouped AS
SELECT
    r.PSID,
    r.Financial_Class,
    r.Payor_Source,
    r.AgencyId,
    r.PPS,
    r.KeyID,
    MIN(r.KeyDate) AS KeyDate,
    MAX(r.EndDate) AS EndDate,
    r.PAID,
    r.ClientName,
    r.PTID,
    r.Payor_Type,
    r.SLID,
    r.slDesc,
    r.Agency,
    r.InsuredId,
    r.EOEType,
    r.BillDate,
    r.Reporting_BranchCode,
    SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0)) AS Revenue,
    SUM(COALESCE(EarnedRev, 0)) AS EarnedRev,
    SUM(COALESCE(Adjustments, 0)) AS Adjustments,
    SUM(COALESCE(UnEarnedRev, 0)) AS UnEarnedRev,
    SUM(COALESCE(Cash, 0)) AS Cash,
    SUM(COALESCE(Credits, 0)) AS Credits,
    SUM(COALESCE(Refunds, 0)) AS Refunds,
    -- Aging Buckets
    COALESCE(SUM(CASE WHEN AgingDays < 0 THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A1,
    COALESCE(SUM(CASE WHEN AgingDays >= {a0} AND AgingDays <= {a1} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A2,
    COALESCE(SUM(CASE WHEN AgingDays > {a1} AND AgingDays <= {a2} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A3,
    COALESCE(SUM(CASE WHEN AgingDays > {a2} AND AgingDays <= {a3} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A4,
    COALESCE(SUM(CASE WHEN AgingDays > {a3} AND AgingDays <= {a4} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A5,
    COALESCE(SUM(CASE WHEN AgingDays > {a4} AND AgingDays <= {a5} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A6,
    COALESCE(SUM(CASE WHEN AgingDays > {a5} AND AgingDays <= {a6} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A7,
    COALESCE(SUM(CASE WHEN AgingDays > {a6} AND AgingDays <= {a7} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A8,
    COALESCE(SUM(CASE WHEN AgingDays > {a7} AND AgingDays <= {a8} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A9,
    COALESCE(SUM(CASE WHEN AgingDays > {a8} AND AgingDays <= {a9} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A10,
    COALESCE(SUM(CASE WHEN AgingDays > {a9} AND AgingDays <= {a10} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A11,
    COALESCE(SUM(CASE WHEN AgingDays > {a10} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A12,
    r.DaysDelayed,
    SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) AS GrossAR,
    SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) AS NetEarnedAR,
    COALESCE(SUM(CASE WHEN r.AgingDays < 0 THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS EpisodeInProgress,
    CASE 
        WHEN SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) < 0 
        THEN SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) 
        ELSE 0 
    END AS CreditsOverPayments,
    '{aging_label1}' as AgingLabel1,
    '{aging_label2}' as AgingLabel2,
    '{aging_label3}' as AgingLabel3,
    '{aging_label4}' as AgingLabel4,
    '{aging_label5}' as AgingLabel5,
    '{aging_label6}' as AgingLabel6,
    '{aging_label7}' as AgingLabel7,
    '{aging_label8}' as AgingLabel8,
    '{aging_label9}' as AgingLabel9,
    '{aging_label10}' as AgingLabel10,
    '{aging_label11}' as AgingLabel11
FROM Report_MonthEndClose_Combined r
WHERE {aging_filter_condition}
  AND (
      ((COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) < {p_exclude_lower})
      OR 
      ((COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) > {p_exclude_upper})
      OR 
      (Credits IS NOT NULL OR Refunds IS NOT NULL)
  )
GROUP BY 
    r.PPS, r.KeyID, r.PAID, r.ClientName, r.PTID, r.Payor_Type, r.SLID, r.slDesc, 
    r.Agency, r.InsuredId, r.EOEType, r.Reporting_BranchCode, r.BillDate, r.DaysDelayed, 
    r.AgencyId, r.PSID, r.Financial_Class, r.Payor_Source
""")

line_item_grouped_count = spark.sql("SELECT COUNT(*) as count FROM Report_LineItem_Grouped").collect()[0]['count']
print(f"✓ Line item grouped report: {line_item_grouped_count} records")

# =============================================
# SECTION 2: Grouped Report (Balance Determined at Client/PS Level)
# For when BalanceDeterminedAt = 1 or 2
# =============================================

print("\nSection 2: Creating grouped report...")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Report_Grouped AS
SELECT
    r.PSID,
    r.Financial_Class,
    r.Payor_Source,
    r.AgencyId,
    r.PPS,
    r.KeyID,
    MIN(r.KeyDate) AS KeyDate,
    MAX(r.EndDate) AS EndDate,
    r.PAID,
    r.ClientName,
    r.PTID,
    r.Payor_Type,
    r.SLID,
    r.slDesc,
    r.Agency,
    r.InsuredId,
    r.EOEType,
    r.BillDate,
    r.Reporting_BranchCode,
    SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0)) AS Revenue,
    SUM(COALESCE(EarnedRev, 0)) AS EarnedRev,
    SUM(COALESCE(Adjustments, 0)) AS Adjustments,
    SUM(COALESCE(UnEarnedRev, 0)) AS UnEarnedRev,
    SUM(COALESCE(Cash, 0)) AS Cash,
    SUM(COALESCE(Credits, 0)) AS Credits,
    SUM(COALESCE(Refunds, 0)) AS Refunds,
    -- Aging Buckets
    COALESCE(SUM(CASE WHEN AgingDays < 0 THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A1,
    COALESCE(SUM(CASE WHEN AgingDays >= {a0} AND AgingDays <= {a1} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A2,
    COALESCE(SUM(CASE WHEN AgingDays > {a1} AND AgingDays <= {a2} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A3,
    COALESCE(SUM(CASE WHEN AgingDays > {a2} AND AgingDays <= {a3} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A4,
    COALESCE(SUM(CASE WHEN AgingDays > {a3} AND AgingDays <= {a4} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A5,
    COALESCE(SUM(CASE WHEN AgingDays > {a4} AND AgingDays <= {a5} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A6,
    COALESCE(SUM(CASE WHEN AgingDays > {a5} AND AgingDays <= {a6} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A7,
    COALESCE(SUM(CASE WHEN AgingDays > {a6} AND AgingDays <= {a7} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A8,
    COALESCE(SUM(CASE WHEN AgingDays > {a7} AND AgingDays <= {a8} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A9,
    COALESCE(SUM(CASE WHEN AgingDays > {a8} AND AgingDays <= {a9} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A10,
    COALESCE(SUM(CASE WHEN AgingDays > {a9} AND AgingDays <= {a10} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A11,
    COALESCE(SUM(CASE WHEN AgingDays > {a10} THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS A12,
    r.DaysDelayed,
    SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) AS GrossAR,
    SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) - COALESCE(Cash, 0)) AS NetEarnedAR,
    COALESCE(SUM(CASE WHEN r.AgingDays < 0 THEN (COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) ELSE 0 END), 0) AS EpisodeInProgress,
    CASE 
        WHEN SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) < 0 
        THEN SUM(COALESCE(EarnedRev, 0) + COALESCE(Adjustments, 0) + COALESCE(UnEarnedRev, 0) - COALESCE(Cash, 0)) 
        ELSE 0 
    END AS CreditsOverPayments,
    '{aging_label1}' as AgingLabel1,
    '{aging_label2}' as AgingLabel2,
    '{aging_label3}' as AgingLabel3,
    '{aging_label4}' as AgingLabel4,
    '{aging_label5}' as AgingLabel5,
    '{aging_label6}' as AgingLabel6,
    '{aging_label7}' as AgingLabel7,
    '{aging_label8}' as AgingLabel8,
    '{aging_label9}' as AgingLabel9,
    '{aging_label10}' as AgingLabel10,
    '{aging_label11}' as AgingLabel11
FROM Report_MonthEndClose_Combined r
WHERE {aging_filter_condition}
GROUP BY 
    r.PPS, r.KeyID, r.PAID, r.ClientName, r.PTID, r.Payor_Type, r.SLID, r.slDesc, 
    r.Agency, r.InsuredId, r.EOEType, r.Reporting_BranchCode, r.BillDate, r.DaysDelayed, 
    r.AgencyId, r.PSID, r.Financial_Class, r.Payor_Source
""")

grouped_count = spark.sql("SELECT COUNT(*) as count FROM Report_Grouped").collect()[0]['count']
print(f"✓ Grouped report: {grouped_count} records")

# =============================================
# SECTION 3: Final Output - HCHBAR Structure
# =============================================

print("\nSection 3: Creating final output table...")

# Using Report_LineItem_Grouped since BalanceDeterminedAt = 3
spark.sql(f"""
CREATE OR REPLACE TABLE {tempdb_dbo_hchbar}
USING DELTA
AS
SELECT
    PPS as pps,
    KeyDate as key_date,
    EndDate as end_date,
    ClientName as client_last_name,
    PAID as paid,
    Payor_Type as payor_type,
    PSID as psid,
    Payor_Source as payor_name,
    BillDate as bill_date,
    Reporting_BranchCode as reporting_branchcode,
    Revenue as revenue,
    EarnedRev as earned_rev,
    UnEarnedRev as unearned_rev,
    Adjustments as adjustments,
    Cash as cash,
    A1 AS episode_in_progress,
    (A2 + A3 + A4 + A5) AS days_0_90,
    (A6 + A7) AS days_91_180,
    A8 AS days_181_270,
    A9 AS days_271_360,
    CAST(0 AS DECIMAL(15,4)) AS days_361_plus,
    GrossAR as gross_ar,
    NetEarnedAR as net_earned_ar,
    '{reporting_week_ending_date}' AS reporting_week_ending_date
FROM Report_LineItem_Grouped
""")

final_count = spark.sql(f"SELECT COUNT(*) as count FROM {tempdb_dbo_hchbar}").collect()[0]['count']